# Real BERT Embedding Compression Experiment
**Paper**: Exploiting Low-Dimensional Manifold Structure for Archival Compression of High-Dimensional ML Embeddings  
**Author**: Francisco Molina-Burgos, Avermex Research Division  
**Purpose**: Validate compression ratios on REAL text (AG News, Wikipedia) — not synthetic templates  
**Runtime**: ~5 min on Colab T4 GPU

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets torch numpy

In [ ]:
# Cell 2 — Load BERT model
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
import json, time, zlib

print(f'GPU available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

print('Loading BERT-base-uncased (768D)...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased').to(device)
bert.train(False)   # inference mode: disables dropout
print('BERT loaded.')

In [ ]:
# Cell 3 — Load REAL datasets
# AG News: real Reuters/AP/BBC news articles, 4 categories
print('Loading AG News (real news articles)...')
ag = load_dataset('ag_news', split='train[:2000]')
news_texts = [row['text'] for row in ag]
print(f'AG News loaded: {len(news_texts)} articles')
print(f'Sample: {news_texts[0][:150]}')

# Wikipedia: wikimedia/wikipedia is the Parquet version — no loading script
print('\nLoading Wikipedia sentences...')
wiki_ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train[:600]')
wiki_texts = []
for article in wiki_ds:
    sentences = [s.strip() for s in article['text'].split('.') if len(s.strip()) > 60]
    wiki_texts.extend(sentences[:4])
    if len(wiki_texts) >= 2000:
        break
wiki_texts = wiki_texts[:2000]
print(f'Wikipedia loaded: {len(wiki_texts)} sentences')
print(f'Sample: {wiki_texts[0][:150]}')

In [ ]:
# Cell 4 — Generate BERT embeddings (batched for speed)
def get_embeddings(texts, batch_size=32):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', truncation=True,
                        max_length=128, padding=True).to(device)
        with torch.no_grad():
            out = bert(**enc)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()  # [CLS] token
        all_emb.append(cls)
        if i % 320 == 0:
            print(f'  {i}/{len(texts)}')
    return np.vstack(all_emb)

print('Generating AG News embeddings...')
t0 = time.time()
news_emb = get_embeddings(news_texts)
print(f'Done: {news_emb.shape} in {time.time()-t0:.1f}s')

print('Generating Wikipedia embeddings...')
t0 = time.time()
wiki_emb = get_embeddings(wiki_texts)
print(f'Done: {wiki_emb.shape} in {time.time()-t0:.1f}s')

In [ ]:
# Cell 5 — Compression + attractor analysis
def consec_sim(emb):
    norms = np.linalg.norm(emb, axis=1)
    return float(np.mean(np.sum(emb[:-1]*emb[1:], axis=1) / (norms[:-1]*norms[1:])))

def cosine_loss(A, B):
    na = np.linalg.norm(A, axis=1, keepdims=True)
    nb = np.linalg.norm(B, axis=1, keepdims=True)
    return float((1 - np.mean(np.sum((A/na)*(B/nb), axis=1))) * 100)

def correlation_dim(emb, n=200):
    X = emb[np.random.choice(len(emb), n, replace=False)]
    dists = []
    for i in range(n):
        for j in range(i+1, n):
            dists.append(float(np.linalg.norm(X[i]-X[j])))
    dists = np.array(dists)
    rs = np.logspace(np.log10(np.percentile(dists,5)), np.log10(np.percentile(dists,75)), 20)
    Cs = np.array([np.mean(dists < r) for r in rs])
    valid = (Cs > 0)
    if valid.sum() < 3: return float('nan')
    return float(np.polyfit(np.log(rs[valid]), np.log(Cs[valid]), 1)[0])

def pca_compress(emb, k):
    mean = emb.mean(0)
    U, S, Vt = np.linalg.svd(emb - mean, full_matrices=False)
    proj = (U[:,:k] * S[:k]).astype(np.float32)
    deltas = np.diff(proj, axis=0, prepend=proj[:1])
    comp_size = len(zlib.compress(deltas.tobytes(), 9))
    recon = (U[:,:k] * S[:k]) @ Vt[:k] + mean
    return n_bytes / comp_size, cosine_loss(emb, recon)

def run(name, emb):
    global n_bytes
    n_bytes = emb.astype(np.float32).nbytes
    raw = emb.astype(np.float32).tobytes()
    print(f'\n{"="*55}')
    print(f'{name}  |  {emb.shape}  |  {n_bytes/1e6:.1f} MB')
    print(f'Consecutive similarity : {consec_sim(emb):.4f}')
    print(f'Correlation dimension D2: {correlation_dim(emb):.3f}')
    print(f'{"="*55}')
    
    gz  = len(zlib.compress(raw, 9))
    dgz = len(zlib.compress(np.diff(emb, axis=0, prepend=emb[:1]).astype(np.float32).tobytes(), 9))
    print(f'GZIP              : {n_bytes/gz:7.2f}x   0.00% loss')
    print(f'Delta+GZIP        : {n_bytes/dgz:7.2f}x   0.00% loss')
    
    results = {'gzip': n_bytes/gz, 'delta_gzip': n_bytes/dgz}
    for k in [5, 10, 20, 50]:
        ratio, loss = pca_compress(emb, k)
        star = ' ★' if ratio > 20 and loss < 20 else ''
        print(f'Attractor(PCA-{k:<3})  : {ratio:7.2f}x  {loss:6.2f}% loss{star}')
        results[f'pca_{k}'] = {'ratio': ratio, 'loss': loss}
    return results

r_news = run('AG News (real Reuters/AP/BBC)', news_emb)
r_wiki = run('Wikipedia (real encyclopedia)', wiki_emb)

In [ ]:
# Cell 6 — Save results for paper
output = {
    'ag_news':    {'consec_sim': consec_sim(news_emb), 'd2': correlation_dim(news_emb), 'compression': r_news},
    'wikipedia':  {'consec_sim': consec_sim(wiki_emb), 'd2': correlation_dim(wiki_emb), 'compression': r_wiki},
}
with open('real_bert_results_final.json', 'w') as f:
    json.dump(output, f, indent=2)

# Download from Colab
from google.colab import files
files.download('real_bert_results_final.json')
print('Downloaded: real_bert_results_final.json')
print('Share this file to update the paper with verified numbers.')